In [ ]:
import requests
import json
import os

In [ ]:
folder = "test_endpoints"
os.makedirs(folder, exist_ok=True)

In [ ]:
def _infer_schema(value): 
    if value is None: 
        return {"type": "null"} 
    if isinstance(value, bool): 
        return {"type": "boolean"} 
    if isinstance(value, int): 
        return {"type": "integer"} 
    if isinstance(value, float): 
        return {"type": "number"} 
    if isinstance(value, str): 
        return {"type": "string"} 
    if isinstance(value, list): 
        schema = {
            "type": "array"
        } 
        if value: 
            item_schemas = [_infer_schema(item) for item in value] 
            if all(item_schema == item_schemas[0] for item_schema in item_schemas):
                schema["items"] = item_schemas[0]
            else:
                schema["items"] = item_schemas
        return schema 
    if isinstance(value, dict): 
        properties = {} 
        for key, child_value in value.items(): 
            properties[key] = _infer_schema(child_value) 
        return { "type": "object", "properties": properties } 
    return { "type": "unknown" }

def explore_endpoint(
    url_v,
    endpoint,
    params=None,
    timeout=30
):
    BASE_URL = {
        "1": "https://statsapi.mlb.com/api/v1/",
        "1.1": "https://statsapi.mlb.com/api/v1.1/"
    }
    if endpoint.startswith("http://") or endpoint.startswith("https://"):
        url = endpoint
    else:
        endpoint = endpoint.lstrip("/")
        url = f"{BASE_URL[url_v]}/{endpoint}"

    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()

    data = response.json()

    return {
        "schema": _infer_schema(data),
        "response": data
    }

In [ ]:
def compare_responses(response1, response2):
    if isinstance(response1, dict) and isinstance(response2, dict):
        keys = {}

        all_keys = response1.keys() | response2.keys()

        for key in all_keys:
            in_1 = key in response1
            in_2 = key in response2

            if in_1 and not in_2:
                keys[key] = False
            elif not in_1 and in_2:
                keys[key] = "added"
            else:
                keys[key] = compare_responses(
                    response1[key],
                    response2[key]
                )

        return keys

    if isinstance(response1, list) and isinstance(response2, list):
        results = []

        for i, (item1, item2) in enumerate(zip(response1, response2)):
            results.append(
                compare_responses(item1, item2)
            )

        if len(response1) > len(response2):
            results.extend(
                [False] * (len(response1) - len(response2))
            )

        elif len(response2) > len(response1):
            results.extend(
                ["added"] * (len(response2) - len(response1))
            )

        return results

    return True

def recount_responses(response, path=""):
    false_count = 0
    added_count = 0
    different_fields = []

    if isinstance(response, dict):
        for key, value in response.items():
            current_path = f"{path}.{key}" if path else key

            false, added, fields = recount_responses(
                value,
                current_path
            )

            false_count += false
            added_count += added
            different_fields.extend(fields)

    elif isinstance(response, list):
        for i, item in enumerate(response):
            current_path = f"{path}[{i}]"

            false, added, fields = recount_responses(
                item,
                current_path
            )

            false_count += false
            added_count += added
            different_fields.extend(fields)

    elif response is False:
        false_count += 1
        different_fields.append({
            "path": path,
            "status": "removed"
        })

    elif response == "added":
        added_count += 1
        different_fields.append({
            "path": path,
            "status": "added"
        })

    return false_count, added_count, different_fields

## Sports

In [ ]:
sports = explore_endpoint(
    url_v="1", 
    endpoint="sports"
)
display(sports['response'])

In [ ]:
# MLB: id = 1 
MLB_SPORT_ID = 1

mlb = explore_endpoint(
    url_v="1", 
    endpoint=f"sports/{MLB_SPORT_ID}"
)
display(mlb['response'])

## Leagues

In [ ]:
league = explore_endpoint(
    url_v="1",
    endpoint="league"
)
display(league['response'])

In [ ]:
# AL: id = 103
AL_LEAGUE_ID = 103

al = explore_endpoint(
    url_v="1",
    endpoint=f"league/{AL_LEAGUE_ID}"
)
display(al['response'])

In [ ]:
# NL: id = 104
NL_LEAGUE_ID = 104

nl = explore_endpoint(
    url_v="1",
    endpoint=f"league/{NL_LEAGUE_ID}"
)
display(nl['response'])

## Divisions

In [ ]:
al_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)
display(al_divisions['response'])

In [ ]:
nl_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)
display(nl_divisions['response'])

## Teams

In [ ]:
al_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)
al_teams_id = {team['name']: team['id'] for team in al_teams['response']['teams']}
display(al_teams['response'])
display(al_teams_id)

In [ ]:
nl_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)
nl_teams_id = {team['name']: team['id'] for team in nl_teams['response']['teams']}
display(nl_teams['response'])
display(nl_teams_id)

## Venue

In [ ]:
pirates_venue_id = 31
pirates_venue = explore_endpoint(
    url_v="1",
    endpoint=f"venues/{pirates_venue_id}",
    params=[
        ("hydrate", "fieldInfo")
    ]
)
display(pirates_venue['response'])

## Roster

In [ ]:
pirates_team_id = 134
pirates_team = explore_endpoint(
    url_v="1",
    endpoint=f"teams/{pirates_team_id}/roster/fullRoster"
)
display(pirates_team['response'])

## Schedule

In [ ]:
schedule = explore_endpoint(
    url_v="1",
    endpoint="schedule",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("date", "2026-08-11")
    ]
)
display(schedule['response'])
with open(f"{folder}/schedule_response.json", "w") as f:
    json.dump(schedule['response'], f, indent=4)

In [ ]:
schedule_hidration = explore_endpoint(
    url_v="1",
    endpoint="schedule",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("date", "2026-08-11"),
        ("hydrate", "team(standings)")
    ]
)
with open(f"{folder}/schedule_hydration_response.json", "w") as f:
    json.dump(schedule_hidration['response'], f, indent=4)

In [ ]:
comparacion = compare_responses(
    schedule['response'],
    schedule_hidration['response']
)
recount_responses(comparacion)

In [ ]:
GAME_PK = schedule['response']['dates'][0]['games'][0]['gamePk']
print(f"Game PK: {GAME_PK}")

## Game

### Content

In [ ]:
content = explore_endpoint(
    url_v="1",
    endpoint=f"game/{GAME_PK}/content"
)
with open(f"{folder}/content_schema.json", "w") as f:
    json.dump(content['schema'], f, indent=4)
with open(f"{folder}/content_response.json", "w") as f:
    json.dump(content['response'], f, indent=4)

### Gumbo

In [ ]:
gumbo = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live"
)
with open(f"{folder}/gumbo_schema.json", "w") as f:
    json.dump(gumbo['schema'], f, indent=4)
with open(f"{folder}/gumbo_response.json", "w") as f:
    json.dump(gumbo['response'], f, indent=4)

## Gumbo hidrations

### credits

In [ ]:
gumbo_hydration_credits = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "credits")
    ]
)
with open(f"{folder}/gumbo_hydration_credits_response.json", "w") as f:
    json.dump(gumbo_hydration_credits['response'], f, indent=4)

In [ ]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_credits['response']
)
recount_responses(comparacion)

### alignment

In [ ]:
gumbo_hydration_alignment = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "alignment")
    ]
)
with open(f"{folder}/gumbo_hydration_alignment_response.json", "w") as f:
    json.dump(gumbo_hydration_alignment['response'], f, indent=4)

In [ ]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_alignment['response']
)
recount_responses(comparacion)

### preState

In [ ]:
gumbo_hydration_preState = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "preState")
    ]
)
with open(f"{folder}/gumbo_hydration_preState_response.json", "w") as f:
    json.dump(gumbo_hydration_preState['response'], f, indent=4)

In [ ]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_preState['response']
)
recount_responses(comparacion)